In [1]:
! git clone https://github.com/zygmuntz/goodbooks-10k.git

Cloning into 'goodbooks-10k'...
Updating files:  65% (13/20)
Updating files:  70% (14/20)
Updating files:  75% (15/20)
Updating files:  80% (16/20)
Updating files:  85% (17/20)
Updating files:  90% (18/20)
Updating files:  95% (19/20)
Updating files: 100% (20/20)
Updating files: 100% (20/20), done.


In [2]:
import pandas as pd
import re

In [3]:
df = pd.read_csv('goodbooks-10k/books.csv')
df.head()

,book_id,goodreads_book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,3198671,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,2683664,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...


In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [5]:
df.columns

Index(['book_id', 'goodreads_book_id', 'best_book_id', 'work_id',
       'books_count', 'isbn', 'isbn13', 'authors', 'original_publication_year',
       'original_title', 'title', 'language_code', 'average_rating',
       'ratings_count', 'work_ratings_count', 'work_text_reviews_count',
       'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5',
       'image_url', 'small_image_url'],
      dtype='str')

In [6]:
df['combined'] = df['title'] + " " + df['authors']
df['cleaned_text'] = df['combined'].apply(clean_text)

In [7]:
df.to_csv("cleaned_books.csv", index=False)

In [8]:
! pip install sentence-transformers


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [9]:
df.columns

Index(['book_id', 'goodreads_book_id', 'best_book_id', 'work_id',
       'books_count', 'isbn', 'isbn13', 'authors', 'original_publication_year',
       'original_title', 'title', 'language_code', 'average_rating',
       'ratings_count', 'work_ratings_count', 'work_text_reviews_count',
       'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5',
       'image_url', 'small_image_url', 'combined', 'cleaned_text'],
      dtype='str')

In [10]:
embeddings = model.encode(df['cleaned_text'].tolist())

In [11]:
import numpy as np

embeddings = np.array(embeddings)
np.save("book_embeddings.npy", embeddings)

In [ ]:
embeddings = np.load('hadith_embeddings.npy')

In [12]:
! pip install faiss-cpu


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import faiss

dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embeddings)

faiss.write_index(faiss_index, "faiss_index.index")

In [14]:
def search_books(query, count=5):
    query = clean_text(query)
    query_embedding = model.encode([query])

    distance, indices = faiss_index.search(query_embedding, count)

    for i in range(count):
        idx = indices[0][i]
        print("Book:", df['title'].iloc[idx])
        print("Author:", df['authors'].iloc[idx])
        print("Distance:", distance[0][i])
        print()

In [15]:
search_books("books about magic wizard")

Book: Harry Potter Schoolbooks Box Set: Two Classic Books from the Library of Hogwarts School of Witchcraft and Wizardry
Author: J.K. Rowling
Distance: 0.82245576

Book: The Magicians' Guild (Black Magician Trilogy, #1)
Author: Trudi Canavan
Distance: 0.8770914

Book: Wizard at Large (Magic Kingdom of Landover, #3)
Author: Terry Brooks
Distance: 0.8891381

Book: The Glass Magician (The Paper Magician Trilogy, #2)
Author: Charlie N. Holmberg
Distance: 0.8912416

Book: Witch & Wizard (Witch & Wizard, #1)
Author: James Patterson, Gabrielle Charbonnet
Distance: 0.895262

